# 45. 数据结构与主题

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 2 / 20 步：建立整洁数据与统计绘图语义**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** Seaborn 模块入门  →  **本章任务：** 数据结构与主题  →  **下一步：** 频数图（countplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。



## 本章目标

学完本章，你将能够：

- **理解**：理解「数据结构与主题」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「数据结构与主题」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「数据结构与主题」并读出其中的结论。


## 45.1 适用场景

**背景引入**：面对一张几十万行的数据表，靠肉眼逐行找规律几乎不可能——我们需要把关系“画”出来。这一章用 Seaborn 教你直接用 DataFrame 完成统计聚合、分类映射和统一视觉风格：数据在表里怎么组织，决定了图画出来能不能让人一眼看懂。先解决“表怎么放”这个前置问题，后面的图表才能又好又省力。

使用DataFrame直接完成统计聚合、分类映射和统一视觉风格。


## 45.2 数据结构

优先使用每行一个观察、每列一个变量的长表。

**打个比方**：长表就像一份班级花名册——每一行是一个学生、每一列是他的一个属性（姓名/年龄/成绩）；宽表则像把全班每个学生的成绩摊成一大张横表。Seaborn 的绘图函数只认‘花名册’，给它横着摊开的宽表，它反而分不清哪一列才是要比较的。


## 45.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 sns.axes_style("ticks") 改为 "whitegrid" 或 "dark"，对比不同主题风格
2. 修改 palette 参数从 "Set2" 为 "pastel" 或 "muted"，观察调色板变化
3. 在 scatterplot 中添加 style="channel" 参数，观察形状映射与颜色映射的组合效果


## 45.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.scatterplot()`、`ax.set()`、`ax.legend()` | 使用DataFrame直接完成统计聚合、分类映射和统一视觉风格。 | 宽表和长表混用 |
| 进阶变体 | `sns.axes_style()`、`plt.subplots()`、`sns.countplot()`、`sns.boxplot()` | 在基础图表上增加分组、注释、布局或交互 | 不知道barplot默认计算均值 |
| 关键参数 | `data` | 数据表 | 宽表和长表混用 |
| 关键参数 | `x/y` | 位置变量 | 不知道barplot默认计算均值 |
| 关键参数 | `hue` | 颜色分组 | Figure级函数传入已有Axes |
| 关键参数 | `style/size` | 其他映射 | 宽表和长表混用 |


## 45.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 45.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.scatterplot(data=marketing, x="visits", y="sales", hue="channel", ax=ax)
ax.set(title="Seaborn长表映射", xlabel="访问量", ylabel="销售额")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


**练一练**：修改图表参数，观察渠道的形状映射`45.6 基础图表` 的散点图默认用 `hue="channel"` 把不同渠道染成不同颜色，`x` 与 `y` 分别是访问量与销售额。请在 `sns.scatterplot` 中补上 `style` 参数，把渠道（`channel`）同时映射到点的**形状**，并运行自检。观察：当颜色和形状都指向渠道时，同一类渠道既"色不同"也"形不同"，图的可区分度更强——这就是在颜色之外再叠加一种形状编码。用一两句话把这种变化记录下来。


In [ ]:
# 请在下方填写代码
import seaborn as sns
import matplotlib.pyplot as plt

# 与本小节示例结构一致：同一份 marketing 长表（x=访问量，y=销售额，颜色=渠道）
# 练一练：把 style 参数从 None 改为渠道字段名 "channel"，让形状也映射到渠道
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.scatterplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",  # 颜色仍表示渠道
    style=None,  # ← 请把 None 改为 "channel"
    ax=ax,
)
ax.set(title="Seaborn长表映射（颜色+形状）", xlabel="访问量", ylabel="销售额")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
# 完整答案：_X_ = "channel"，把渠道同时映射为颜色与形状
sns.scatterplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    style="channel",
    ax=ax,
)
ax.set(
    title="Seaborn长表映射（颜色+形状·完整答案）",
    xlabel="访问量",
    ylabel="销售额",
)
ax.legend(title="渠道", frameon=False)
fig.tight_layout()


## 45.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

with sns.axes_style("ticks"):
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.countplot(data=orders, x="category", color="#1a73e8", ax=axes[0])
    sns.boxplot(
        data=orders,
        x="category",
        y="order_value",
        hue="category",
        palette="Set2",
        legend=False,
        ax=axes[1],
    )
    axes[0].set(title="订单量", xlabel="品类", ylabel="订单数")
    axes[1].set(title="客单价分布", xlabel="品类", ylabel="元")
    sns.despine()
    fig.tight_layout()
plt.show()


## 45.8 参数说明

- data：数据表
- x/y：位置变量
- hue：颜色分组
- style/size：其他映射


## 45.9 结果解读

先确认每个视觉通道对应哪一列，再判断函数是否自动执行了统计聚合。


## 45.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 45.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 45.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 45.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 45.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 45.12 易错点提醒

- 宽表和长表混用
- 不知道barplot默认计算均值
- Figure级函数传入已有Axes


## 45.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 45.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：换一种主题风格，观察同一张图的观感差异
# 【目标】seaborn 的主题(style)决定背景、网格、边框等整体观感。练习切换并对比。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：把 set_theme 的 style 换成 "darkgrid"(深色网格)。
sns.set_theme(style="darkgrid", context="notebook")
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.scatterplot(data=marketing, x="visits", y="sales", hue="channel", ax=ax)
ax.set(title="Seaborn长表映射（darkgrid）", xlabel="访问量", ylabel="销售额")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：切换主题后，图的观感与可读性如何变化 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=orders, x="category", y="order_value", ci=None, ax=ax)
ax.set(title="品类平均客单价", xlabel="品类", ylabel="元")
fig.tight_layout()
plt.show()


## 45.15 小结

理解Seaborn的长表映射、Axes级与Figure级接口、主题和调色板。


### 45.15.1 你已经掌握

- 判断数据结构与主题的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 45.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `data` | 数据表 |
| `x/y` | 位置变量 |
| `hue` | 颜色分组 |
| `style/size` | 其他映射 |


### 45.15.3 需要注意

- 宽表和长表混用
- 不知道barplot默认计算均值
- Figure级函数传入已有Axes


### 45.15.4 完成检查

- [ ] 能判断什么问题适合使用数据结构与主题
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 45.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
